# 📈 Multi-Stock Price Prediction
### Comparing 6 AI Models: LSTM · Bidirectional LSTM · GRU · CNN-LSTM · Linear Regression · Random Forest

**Stocks:** AAPL, NVDA, MSFT, GOOGL  
**Period:** 2015–2023  
**Prediction:** Future 1 Year


## 1️⃣ Imports

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, LSTM, GRU,
    Bidirectional, Conv1D,
    MaxPooling1D, Flatten, Input
)


## 2️⃣ Settings

In [ ]:
stocks     = ["AAPL", "NVDA", "MSFT", "GOOGL"]
start_date = "2015-01-01"
end_date   = "2023-12-31"
time_step  = 60
future_days = 365


## 3️⃣ Helper Functions

In [ ]:
def prepare_data(df, time_step=60):
    """Scale data and create X, y sequences."""
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df.values)

    X, y = [], []
    for i in range(time_step, len(scaled)):
        X.append(scaled[i - time_step:i, 0])
        y.append(scaled[i, 0])

    X = np.array(X)
    y = np.array(y)
    return X, y, scaler, scaled


def train_test_split_ts(X, y, ratio=0.8):
    """Split time-series data (no shuffle)."""
    split = int(len(X) * ratio)
    return X[:split], X[split:], y[:split], y[split:]


def calc_metrics(real, pred):
    """Return MAE, RMSE, R2, Accuracy%."""
    mae  = mean_absolute_error(real, pred)
    rmse = np.sqrt(mean_squared_error(real, pred))
    r2   = r2_score(real, pred)
    acc  = 100 - (mae / np.mean(real) * 100)
    return mae, rmse, r2, acc


def predict_future(model_fn, last_scaled, scaler, future_days=365, time_step=60):
    """Predict future days using rolling window (for deep-learning models)."""
    temp   = last_scaled[-time_step:].tolist()
    future = []
    for _ in range(future_days):
        x    = np.array(temp[-time_step:]).reshape(1, time_step, 1)
        pred = model_fn(x)
        temp.append([[pred]])
        future.append(pred)
    return scaler.inverse_transform(np.array(future).reshape(-1, 1))


---
## 🔵 Model 1: LSTM (Original)
> **Long Short-Term Memory** — the original model in this project.  
> Good at remembering long-term patterns in time-series data.


In [ ]:
def build_lstm(time_step):
    model = Sequential([
        LSTM(50, return_sequences=True, input_shape=(time_step, 1)),
        LSTM(50),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


lstm_results = []

for symbol in stocks:
    print(f"\n[LSTM] Processing {symbol} ...")

    df = yf.download(symbol, start=start_date, end=end_date)[['Close']]
    df.dropna(inplace=True)

    X, y, scaler, scaled = prepare_data(df, time_step)
    X = X.reshape(X.shape[0], X.shape[1], 1)
    X_train, X_test, y_train, y_test = train_test_split_ts(X, y)

    model = build_lstm(time_step)
    model.fit(X_train, y_train, epochs=5, batch_size=32, verbose=0)

    pred  = scaler.inverse_transform(model.predict(X_test))
    real  = scaler.inverse_transform(y_test.reshape(-1, 1))
    mae, rmse, r2, acc = calc_metrics(real, pred)

    lstm_results.append([symbol, "LSTM", mae, rmse, r2, acc])
    print(f"  MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}  Acc={acc:.2f}%")

print("\n✅ LSTM done")


---
## 🟣 Model 2: Bidirectional LSTM
> Reads the sequence **forward and backward** — captures more patterns  
> and usually gives better accuracy than plain LSTM.


In [ ]:
def build_bilstm(time_step):
    model = Sequential([
        Bidirectional(LSTM(50, return_sequences=True), input_shape=(time_step, 1)),
        Bidirectional(LSTM(50)),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


bilstm_results = []

for symbol in stocks:
    print(f"\n[BiLSTM] Processing {symbol} ...")

    df = yf.download(symbol, start=start_date, end=end_date)[['Close']]
    df.dropna(inplace=True)

    X, y, scaler, scaled = prepare_data(df, time_step)
    X = X.reshape(X.shape[0], X.shape[1], 1)
    X_train, X_test, y_train, y_test = train_test_split_ts(X, y)

    model = build_bilstm(time_step)
    model.fit(X_train, y_train, epochs=5, batch_size=32, verbose=0)

    pred  = scaler.inverse_transform(model.predict(X_test))
    real  = scaler.inverse_transform(y_test.reshape(-1, 1))
    mae, rmse, r2, acc = calc_metrics(real, pred)

    bilstm_results.append([symbol, "BiLSTM", mae, rmse, r2, acc])
    print(f"  MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}  Acc={acc:.2f}%")

print("\n✅ Bidirectional LSTM done")


---
## 🟢 Model 3: GRU (Gated Recurrent Unit)
> Lighter and faster than LSTM, almost the same accuracy.  
> A great alternative when training time matters.


In [ ]:
def build_gru(time_step):
    model = Sequential([
        GRU(50, return_sequences=True, input_shape=(time_step, 1)),
        GRU(50),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


gru_results = []

for symbol in stocks:
    print(f"\n[GRU] Processing {symbol} ...")

    df = yf.download(symbol, start=start_date, end=end_date)[['Close']]
    df.dropna(inplace=True)

    X, y, scaler, scaled = prepare_data(df, time_step)
    X = X.reshape(X.shape[0], X.shape[1], 1)
    X_train, X_test, y_train, y_test = train_test_split_ts(X, y)

    model = build_gru(time_step)
    model.fit(X_train, y_train, epochs=5, batch_size=32, verbose=0)

    pred  = scaler.inverse_transform(model.predict(X_test))
    real  = scaler.inverse_transform(y_test.reshape(-1, 1))
    mae, rmse, r2, acc = calc_metrics(real, pred)

    gru_results.append([symbol, "GRU", mae, rmse, r2, acc])
    print(f"  MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}  Acc={acc:.2f}%")

print("\n✅ GRU done")


---
## 🟡 Model 4: CNN + LSTM
> **CNN** extracts local patterns from the price sequence,  
> then **LSTM** learns the time dependencies — a powerful combo.


In [ ]:
def build_cnn_lstm(time_step):
    model = Sequential([
        Conv1D(filters=64, kernel_size=3, activation='relu',
               input_shape=(time_step, 1)),
        MaxPooling1D(pool_size=2),
        LSTM(50, return_sequences=False),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


cnnlstm_results = []

for symbol in stocks:
    print(f"\n[CNN-LSTM] Processing {symbol} ...")

    df = yf.download(symbol, start=start_date, end=end_date)[['Close']]
    df.dropna(inplace=True)

    X, y, scaler, scaled = prepare_data(df, time_step)
    X = X.reshape(X.shape[0], X.shape[1], 1)
    X_train, X_test, y_train, y_test = train_test_split_ts(X, y)

    model = build_cnn_lstm(time_step)
    model.fit(X_train, y_train, epochs=5, batch_size=32, verbose=0)

    pred  = scaler.inverse_transform(model.predict(X_test))
    real  = scaler.inverse_transform(y_test.reshape(-1, 1))
    mae, rmse, r2, acc = calc_metrics(real, pred)

    cnnlstm_results.append([symbol, "CNN-LSTM", mae, rmse, r2, acc])
    print(f"  MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}  Acc={acc:.2f}%")

print("\n✅ CNN-LSTM done")


---
## 🟠 Model 5: Linear Regression
> Classic baseline model from **sklearn**.  
> Simple but useful for comparing against deep learning models.


In [ ]:
lr_results = []

for symbol in stocks:
    print(f"\n[LinearReg] Processing {symbol} ...")

    df = yf.download(symbol, start=start_date, end=end_date)[['Close']]
    df.dropna(inplace=True)

    X, y, scaler, scaled = prepare_data(df, time_step)
    # For sklearn models: X stays 2D (no reshape to 3D)
    X_train, X_test, y_train, y_test = train_test_split_ts(X, y)

    model = LinearRegression()
    model.fit(X_train, y_train)

    pred  = scaler.inverse_transform(model.predict(X_test).reshape(-1, 1))
    real  = scaler.inverse_transform(y_test.reshape(-1, 1))
    mae, rmse, r2, acc = calc_metrics(real, pred)

    lr_results.append([symbol, "LinearRegression", mae, rmse, r2, acc])
    print(f"  MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}  Acc={acc:.2f}%")

print("\n✅ Linear Regression done")


---
## 🔴 Model 6: Random Forest
> Ensemble model that builds many decision trees.  
> Often beats linear models on non-linear data like stock prices.


In [ ]:
rf_results = []

for symbol in stocks:
    print(f"\n[RandomForest] Processing {symbol} ...")

    df = yf.download(symbol, start=start_date, end=end_date)[['Close']]
    df.dropna(inplace=True)

    X, y, scaler, scaled = prepare_data(df, time_step)
    X_train, X_test, y_train, y_test = train_test_split_ts(X, y)

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    pred  = scaler.inverse_transform(model.predict(X_test).reshape(-1, 1))
    real  = scaler.inverse_transform(y_test.reshape(-1, 1))
    mae, rmse, r2, acc = calc_metrics(real, pred)

    rf_results.append([symbol, "RandomForest", mae, rmse, r2, acc])
    print(f"  MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}  Acc={acc:.2f}%")

print("\n✅ Random Forest done")


---
## 📊 Model Comparison Table
> All 6 models side-by-side for every stock.


In [ ]:
all_results = (
    lstm_results +
    bilstm_results +
    gru_results +
    cnnlstm_results +
    lr_results +
    rf_results
)

comparison_df = pd.DataFrame(
    all_results,
    columns=["Stock", "Model", "MAE", "RMSE", "R2", "Accuracy %"]
)

comparison_df = comparison_df.sort_values(["Stock", "Accuracy %"], ascending=[True, False])
comparison_df = comparison_df.reset_index(drop=True)

# Round numbers
for col in ["MAE", "RMSE", "R2", "Accuracy %"]:
    comparison_df[col] = comparison_df[col].round(4)

comparison_df


---
## 🏆 Best Model Per Stock


In [ ]:
best_df = comparison_df.loc[
    comparison_df.groupby("Stock")["Accuracy %"].idxmax()
].reset_index(drop=True)

print("🏆 Best model for each stock:\n")
print(best_df[["Stock", "Model", "Accuracy %"]].to_string(index=False))


---
## 📈 Accuracy Chart — All Models vs All Stocks


In [ ]:
models = comparison_df["Model"].unique()
stock_list = comparison_df["Stock"].unique()

x      = np.arange(len(stock_list))
width  = 0.13
colors = ["#4C72B0", "#8B4FCF", "#55A868", "#C4A82D", "#E07B3A", "#C44E52"]

fig, ax = plt.subplots(figsize=(14, 6))

for i, (model, color) in enumerate(zip(models, colors)):
    accs = []
    for stock in stock_list:
        row = comparison_df[
            (comparison_df["Model"] == model) &
            (comparison_df["Stock"] == stock)
        ]["Accuracy %"]
        accs.append(row.values[0] if len(row) else 0)

    offset = (i - len(models) / 2) * width + width / 2
    bars = ax.bar(x + offset, accs, width, label=model, color=color, alpha=0.85)

ax.set_xlabel("Stock", fontsize=12)
ax.set_ylabel("Accuracy %", fontsize=12)
ax.set_title("Model Accuracy Comparison per Stock", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(stock_list, fontsize=11)
ax.legend(title="Model", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.set_ylim(0, 110)
ax.axhline(y=100, color="gray", linestyle="--", alpha=0.4)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("model_accuracy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved!")


---
## 🔮 Future Price Prediction — LSTM (1 Year)
> Same as original project: real price (blue) + future prediction (orange).


In [ ]:
for symbol in stocks:

    df = yf.download(symbol, start=start_date, end=end_date)[['Close']]
    df.dropna(inplace=True)

    X, y, scaler, scaled = prepare_data(df, time_step)
    X = X.reshape(X.shape[0], X.shape[1], 1)

    model = build_lstm(time_step)
    model.fit(X, y, epochs=5, batch_size=32, verbose=0)

    # Rolling future prediction
    temp   = scaled[-time_step:].tolist()
    future = []
    for _ in range(future_days):
        x_in = np.array(temp[-time_step:]).reshape(1, time_step, 1)
        p    = model.predict(x_in, verbose=0)[0][0]
        temp.append([p])
        future.append([p])

    future_prices = scaler.inverse_transform(np.array(future))

    # Plot
    future_plot          = np.full((len(df) + future_days, 1), np.nan)
    future_plot[len(df):] = future_prices

    plt.figure(figsize=(12, 4))
    plt.plot(df.values,   label="Real Price",   color="#4C72B0")
    plt.plot(future_plot, label="Future (LSTM)", color="#E07B3A")
    plt.title(f"{symbol} — Real vs Future Price (LSTM)", fontsize=13, fontweight="bold")
    plt.xlabel("Days")
    plt.ylabel("Price (USD)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


---
## 📝 Model Evaluation Summary

| Model | Type | Notes |
|-------|------|-------|
| **LSTM** | Deep Learning | Original model — good baseline |
| **Bidirectional LSTM** | Deep Learning | Reads forward & backward — often more accurate |
| **GRU** | Deep Learning | Faster than LSTM, similar accuracy |
| **CNN-LSTM** | Deep Learning | Extracts local patterns then learns time dependencies |
| **Linear Regression** | Machine Learning | Simple baseline — fast to train |
| **Random Forest** | Machine Learning | Ensemble method — strong on non-linear data |

**Metrics used:**
- **MAE** — Mean Absolute Error (lower is better)  
- **RMSE** — Root Mean Square Error (lower is better)  
- **R²** — How well the model fits the data (closer to 1 is better)  
- **Accuracy %** — Custom formula: `100 - (MAE / mean_price × 100)`

**Stocks:** AAPL · NVDA · MSFT · GOOGL  
**Dataset:** Yahoo Finance · Period: 2015–2023 · Prediction: 1 year ahead
